<div style="background:linear-gradient(135deg,#1a237e 0%,#4527a0 50%,#00695c 100%);padding:40px;border-radius:16px;color:white;text-align:center">
<h1 style="font-size:2.4em;font-weight:900;margin-bottom:8px">Zero-Cost Neural Architecture Search</h1>
<h2 style="font-size:1.3em;font-weight:300;opacity:.85;margin-bottom:20px">for Heterogeneous NCHW Classification Tensors</h2>
<hr style="border-color:rgba(255,255,255,.3);margin:20px 0">
<p style="font-size:.95em;opacity:.75;max-width:700px;margin:0 auto">
We present a training-free architecture search engine that operates over a rich,
geometry-aware search space of <strong>11 primitive blocks</strong> and <strong>7 head types</strong>.
Architecture quality is estimated in seconds using <strong>AZ-NAS</strong> — a zero-cost proxy
combining Expressivity, Progressivity, Trainability and Complexity — without any weight training.
Search is driven by <strong>Aging Evolution</strong>, sampling hundreds of candidates within a time budget
that adapts to each dataset.
</p></div>

---
## ◆ Section 0 — Setup
Install deps, configure paths, define the visual style.

In [ ]:
import subprocess, sys, os, json, warnings, copy, random, time, itertools
warnings.filterwarnings("ignore")

# ── Detect environment ───────────────────────────────────────────────────────
try:
    import google.colab; IN_COLAB = True
    subprocess.check_call(["git","clone","--depth=1",
        "https://github.com/jesusllg/UnseenNAS-AutoML-Competition","/content/repo"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    REPO_ROOT = "/content/repo"
except ImportError:
    IN_COLAB = False
    REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))

SUBMISSION_DIR = os.path.join(REPO_ROOT, "submission")
if SUBMISSION_DIR not in sys.path: sys.path.insert(0, SUBMISSION_DIR)

subprocess.check_call([sys.executable,"-m","pip","install","-q",
    "torch","numpy","matplotlib","seaborn","scikit-learn","scipy"],
    stdout=subprocess.DEVNULL)

import torch, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import accuracy_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓  PyTorch {torch.__version__}  |  device: {DEVICE}")
if DEVICE.type == "cuda": print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"✓  Repo: {REPO_ROOT}")

In [ ]:
# ── Paper-grade visual style ─────────────────────────────────────────────────
PALETTE = dict(
    blue="#1565C0", orange="#E65100", teal="#00695C",
    purple="#4527A0", red="#B71C1C", green="#2E7D32",
    grey="#546E7A", gold="#F9A825", pink="#AD1457", dark="#212121",
)
BLOCK_COLORS = {
    "ConvBlock":"#1565C0","SepConvBlock":"#0277BD","ResidualBlock":"#00695C",
    "MBConvBlock":"#2E7D32","BottleneckBlock":"#E65100","AnisotropicBlock":"#AD1457",
    "DilatedConvBlock":"#6A1B9A","GridLogicBlock":"#4527A0","ChannelMixingBlock":"#00838F",
    "GlobalContextBlock":"#F9A825","LightAttentionBlock":"#C62828",
}
plt.rcParams.update({
    "figure.facecolor":"white","axes.facecolor":"#FAFAFA",
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.color":"#E0E0E0","grid.linewidth":0.6,
    "font.size":11,"axes.titlesize":13,"axes.titleweight":"bold",
    "figure.titlesize":15,"figure.titleweight":"bold",
})
def banner(title, subtitle="", color=None):
    color = color or PALETTE["blue"]
    fig,ax = plt.subplots(figsize=(14,.9)); fig.patch.set_facecolor(color)
    ax.set_facecolor(color); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
    ax.text(.02,.68,title,color="white",fontsize=14,fontweight="bold",va="center")
    ax.text(.02,.22,subtitle,color="white",fontsize=9.5,va="center",alpha=.8)
    plt.tight_layout(pad=0); plt.show()
print("✓  Style configured.")

---
## ◆ Section 1 — Real Data Loading

We load a real competition dataset. If not present we generate structured synthetic stand-ins.
The loader tries common paths automatically — just point it at your `datasets/` folder.

| Dataset | Shape | Classes | Note |
|---|---|---|---|
| **AddNIST** | (N,3,28,28) | 20 | Two overlaid MNIST digits, label = sum |
| **Chesseract** | (N,12,8,8) | 64 | Chess board, 12-channel 8×8 |
| **Voxel** | (N,20,20,20) | 10 | 3-D volumetric data |

In [ ]:
MAX_SAMPLES = 512   # images to load — keep small for fast notebook execution

def try_load_dataset(base_dirs, prefer=("AddNIST","Chesseract")):
    for base in base_dirs:
        if not os.path.isdir(base): continue
        for name in list(prefer)+[d for d in os.listdir(base) if d not in prefer]:
            p = os.path.join(base, name)
            if os.path.exists(os.path.join(p,"train_x.npy")):
                x = np.load(os.path.join(p,"train_x.npy"))[:MAX_SAMPLES]
                y = np.load(os.path.join(p,"train_y.npy"))[:MAX_SAMPLES]
                with open(os.path.join(p,"metadata")) as f: meta = json.load(f)
                return x, y, meta, name
    return None

def make_synthetic(shape=(512,3,28,28), num_classes=20, name="Synthetic-AddNIST"):
    np.random.seed(0); N,C,H,W = shape
    x = np.random.rand(*shape).astype(np.float32)
    # add class-dependent blobs to make it non-trivial
    y = np.repeat(np.arange(num_classes), N//num_classes+1)[:N]
    for i in range(N):
        c = y[i]; cx,cy = (c%(W//2))+4, (c//(W//2))*4+4
        for r in range(min(H,28)):
            for col in range(min(W,28)):
                x[i,0,r%H,col%W] += np.exp(-((r-cy)**2+(col-cx)**2)/6.)*.6
    x = np.clip(x,0,1)
    meta = {"num_classes":num_classes,"input_shape":list(shape),
            "codename":name,"benchmark":50.0,"time_limit":0.1}
    return x, y.astype(np.int64), meta, name

SEARCH_DIRS = [os.path.join(REPO_ROOT,"datasets"), os.path.join(REPO_ROOT,"..","datasets"),
               "/content/datasets", "datasets"]
result = try_load_dataset(SEARCH_DIRS)
if result:
    train_x, train_y, METADATA, DS_NAME = result; DATA_SOURCE = "real"
    print(f"✓  REAL dataset loaded: {DS_NAME}")
else:
    train_x, train_y, METADATA, DS_NAME = make_synthetic(); DATA_SOURCE = "synthetic"
    print(f"⚠  No dataset found — synthetic stand-in ({DS_NAME})")
    print(f"   Run:  python download_datasets.py AddNIST   for real data")

if train_x.ndim == 3: train_x = train_x[:,np.newaxis]
N, C, H, W = train_x.shape
NUM_CLASSES = METADATA["num_classes"]
BENCHMARK   = METADATA.get("benchmark", 50.0)
print(f"   Shape: {train_x.shape}  |  Classes: {NUM_CLASSES}  |  Benchmark: {BENCHMARK}%")

In [ ]:
banner("§1  Dataset Exploration",
       f"{DS_NAME}  ·  {N} samples  ·  {C}×{H}×{W}  ·  {NUM_CLASSES} classes",
       PALETTE["blue"])

fig = plt.figure(figsize=(16,8))
gs  = gridspec.GridSpec(2,2,figure=fig,hspace=.5,wspace=.35,width_ratios=[1.8,1])

# ── Image gallery ────────────────────────────────────────────────────────────
n_show = min(NUM_CLASSES,20); n_cols = min(10,n_show); n_rows = int(np.ceil(n_show/n_cols))
inner  = gridspec.GridSpecFromSubplotSpec(n_rows,n_cols,subplot_spec=gs[:,0],hspace=.08,wspace=.08)
shown  = 0
for cls in range(n_show):
    idxs = np.where(train_y==cls)[0]
    if not len(idxs): continue
    img = train_x[idxs[0]]; img = (img-img.min())/(img.max()-img.min()+1e-8)
    r,col = divmod(shown,n_cols)
    ax = fig.add_subplot(inner[r,col])
    if img.shape[0]==1: ax.imshow(img[0],cmap="inferno",interpolation="nearest")
    elif img.shape[0]==3: ax.imshow(np.transpose(img,(1,2,0)),interpolation="nearest")
    else: ax.imshow(img[0],cmap="viridis",interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f"cls {cls}",fontsize=7,pad=2)
    for sp in ax.spines.values(): sp.set_linewidth(.5); sp.set_color("#BDBDBD")
    shown += 1

# ── Class distribution ───────────────────────────────────────────────────────
ax_d = fig.add_subplot(gs[0,1])
counts = [np.sum(train_y==c) for c in range(NUM_CLASSES)]
colors = plt.cm.rainbow(np.linspace(0,1,NUM_CLASSES))
ax_d.bar(range(NUM_CLASSES),counts,color=colors,edgecolor="white",lw=.5)
ax_d.axhline(np.mean(counts),color=PALETTE["orange"],ls="--",lw=1.5,
             label=f"mean={np.mean(counts):.0f}")
ax_d.set_xlabel("Class"); ax_d.set_ylabel("Samples")
ax_d.set_title("Class Distribution"); ax_d.legend(fontsize=9)

# ── Pixel intensity histogram ─────────────────────────────────────────────────
ax_h = fig.add_subplot(gs[1,1])
CH_COL = ["#E53935","#43A047","#1E88E5","#FB8C00","#8E24AA","#00ACC1"]
flat = train_x.reshape(N,C,-1)
for ch in range(min(C,3)):
    v = flat[:,ch,:].ravel()
    ax_h.hist(v,bins=60,alpha=.65,color=CH_COL[ch],
              label=f"ch {ch}  μ={v.mean():.2f}  σ={v.std():.2f}",density=True)
ax_h.set_xlabel("Pixel value"); ax_h.set_ylabel("Density")
ax_h.set_title("Pixel Intensity Distributions"); ax_h.legend(fontsize=8)

fig.suptitle(f"Dataset: {DS_NAME}   ({DATA_SOURCE} data)",fontsize=14,fontweight="bold",y=1.01)
plt.savefig("fig1_dataset.png",dpi=150,bbox_inches="tight"); plt.show()

---
## ◆ Section 2 — Geometry-Aware Family Detection

Our `infer_family(C, H, W, num_classes)` translates raw tensor dimensions into
hard architectural constraints — without labels, without training.

| Family | Trigger | Max pools | GroupNorm | Attention |
|---|---|---|---|---|
| `anisotropic` | one dim ≥ 6× other | 2 | ✓ | small only |
| `small_grid` | max(H,W) ≤ 10 | 1 | ✓ | ✓ |
| `possible_voxel` | cube-like C≈H≈W | 2 | ✓ | small only |
| `channel_heavy` | C ≥ 8, area ≤ 1024 | 2 | maybe | small only |
| `spatiotemporal_like` | H=1 or W=1 | 3 | – | ✓ |
| `visual_large` | min(H,W) ≥ 64 | 5 | – | small only |
| `visual_medium` | min(H,W) ≥ 24 | 3 | – | ✓ |
| `compact_general` | everything else | 2 | – | small only |

In [ ]:
from search_space import infer_family

KNOWN = {
    "Cryptic (1×6×768)":    (1, 6, 768,20),
    "Chesseract (12×8×8)":  (12,8,  8, 64),
    "Voxel (20×20×20)":     (20,20,20, 10),
    "Windspeed (8×16×16)":  (8, 16,16, 30),
    "Language (3×1×64)":    (3, 1, 64, 10),
    "Myofibre (3×128×128)": (3,128,128,10),
    "AddNIST (3×28×28)":    (3, 28,28, 20),
    "Sudoku (1×9×9)":       (1, 9,  9,  9),
}
FAM_COLORS = {
    "anisotropic":PALETTE["pink"],"small_grid":PALETTE["teal"],
    "possible_voxel":PALETTE["blue"],"channel_heavy":PALETTE["orange"],
    "spatiotemporal_like":PALETTE["purple"],"visual_large":PALETTE["green"],
    "visual_medium":PALETTE["gold"],"compact_general":PALETTE["grey"],
}
families = [infer_family(*v) for v in KNOWN.values()]

banner("§2  Family Detection","Tensor geometry → hard architectural constraints",PALETTE["teal"])

fig,axes = plt.subplots(1,2,figsize=(16,5.5))

# Left: family bar ─────────────────────────────────────────────────────────
ax = axes[0]
labels = list(KNOWN.keys())
y_pos = np.arange(len(labels))
for i,(lbl,fam) in enumerate(zip(labels,families)):
    col = FAM_COLORS[fam.name]
    ax.barh(i,1,color=col,edgecolor="white",height=.72)
    ax.text(.03,i,fam.name.replace("_"," "),va="center",fontsize=9,
            color="white",fontweight="bold")
ax.set_yticks(y_pos); ax.set_yticklabels(labels,fontsize=9)
ax.set_xticks([]); ax.set_xlim(0,1.4)
ax.set_title("Dataset → Inferred Family")
patches=[mpatches.Patch(color=c,label=n.replace("_"," ")) for n,c in FAM_COLORS.items()]
ax.legend(handles=patches,fontsize=7.5,loc="lower right")

# Right: constraint heatmap ──────────────────────────────────────────────────
ax2 = axes[1]
cols_h = ["max\npools","group\nnorm","attention","anisotropic"]
mat = np.array([[f.max_pool_steps/5,int(f.force_groupnorm),
                 int(f.enable_attention),int(f.is_anisotropic)] for f in families])
im = ax2.imshow(mat,aspect="auto",cmap="YlOrRd",vmin=0,vmax=1)
ax2.set_xticks(range(4)); ax2.set_xticklabels(cols_h,fontsize=9)
ax2.set_yticks(range(len(labels)))
ax2.set_yticklabels([l.split("(")[0].strip() for l in labels],fontsize=9)
ax2.set_title("Constraint Matrix")
raw_vals = [[f.max_pool_steps,int(f.force_groupnorm),
             int(f.enable_attention),int(f.is_anisotropic)] for f in families]
for i in range(len(labels)):
    for j in range(4):
        ax2.text(j,i,str(raw_vals[i][j]),ha="center",va="center",
                 fontsize=9,fontweight="bold",color="white" if mat[i,j]>.45 else "#333")
plt.colorbar(im,ax=ax2,shrink=.8,label="normalized")

OUR_FAM = infer_family(C,H,W,NUM_CLASSES)
print(f"Our dataset ({DS_NAME}) → family: {OUR_FAM.name}")
print(f"  max_pool_steps : {OUR_FAM.max_pool_steps}")
print(f"  force_groupnorm: {OUR_FAM.force_groupnorm}")
print(f"  enable_attention:{OUR_FAM.enable_attention}")
print(f"  forbidden_blocks:{OUR_FAM.forbidden_blocks}")
fig.suptitle("Geometry-Aware Family Inference",fontsize=14,fontweight="bold",y=1.02)
plt.savefig("fig2_families.png",dpi=150,bbox_inches="tight"); plt.show()

---
## ◆ Section 3 — The Search Space

Every architecture is a **Genotype**: a dict of categorical genes. The macro skeleton is fixed
(Stem → Stages → Neck → Head) but every component is searched independently.

```
INPUT  →  STEM  →  [STAGE 1  ↓]  →  [STAGE 2  ↓]  →  ...  →  [STAGE N]  →  NECK  →  HEAD  →  logits
```

Each stage = stack of 1–4 identical blocks + optional spatial downsampling.

In [ ]:
from search_space import (
    sample_random_genotype, repair, build_model,
    BLOCK_TYPES, CHANNEL_LIST, HEAD_TYPES, STEM_TYPES, estimate_params,
)
from search_space.genotype import KERNEL_LIST

random.seed(42)
archs = []
for _ in range(5):
    g = sample_random_genotype()
    g = repair(g,C,H,W,NUM_CLASSES,OUR_FAM)
    archs.append(g)

banner("§3  Search Space","11 blocks · 7 heads · pure categorical encoding",PALETTE["purple"])

fig,axes = plt.subplots(5,1,figsize=(15,7.5),gridspec_kw={"hspace":.08})
for ai,(g,ax) in enumerate(zip(archs,axes)):
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
    # global genes (stem)
    for xi,(lbl,col) in enumerate([
        (g.stem_type[:5],PALETTE["grey"]),
        (f"C₀={CHANNEL_LIST[g.stem_channels]}",PALETTE["grey"]),
        (g.norm_type,PALETTE["grey"]),
    ]):
        x0 = .01+xi*.065
        ax.add_patch(mpatches.FancyBboxPatch((x0,.1),.06,.8,
            boxstyle="round,pad=.01",facecolor=col,edgecolor="white",lw=1))
        ax.text(x0+.03,.55,lbl,ha="center",va="center",fontsize=7,color="white",fontweight="bold")
    # stage genes
    x = .21; sw = .56/g.n_stages
    for si,stage in enumerate(g.active_stages):
        col = BLOCK_COLORS.get(stage.block_type,"#888")
        ax.add_patch(mpatches.FancyBboxPatch((x,.05),sw-.008,.9,
            boxstyle="round,pad=.01",facecolor=col,edgecolor="white",lw=1.5))
        short = stage.block_type.replace("Block","").replace("Residual","Res")[:6]
        ax.text(x+sw/2-.004,.72,short,ha="center",va="center",
                fontsize=6.5,color="white",fontweight="bold")
        ax.text(x+sw/2-.004,.47,f"C={CHANNEL_LIST[stage.channels_idx]}",
                ha="center",va="center",fontsize=6,color="rgba(255,255,255,.8)")
        ax.text(x+sw/2-.004,.27,f"k={KERNEL_LIST[stage.kernel_idx]}",
                ha="center",va="center",fontsize=6,color="rgba(255,255,255,.8)")
        x += sw
    # head
    ax.add_patch(mpatches.FancyBboxPatch((x+.01,.1),.10,.8,
        boxstyle="round,pad=.01",facecolor=PALETTE["orange"],edgecolor="white",lw=1))
    ax.text(x+.06,.55,g.head_type[:8],ha="center",va="center",
            fontsize=6.5,color="white",fontweight="bold")
    p = estimate_params(g,C,H,W,NUM_CLASSES)
    ax.text(.995,.55,f"{p/1e6:.1f}M params",ha="right",va="center",fontsize=8,color="#555")
    ax.text(-.002,.55,f"#{ai+1}",ha="right",va="center",fontsize=9,fontweight="bold",color="#333")

patches = [mpatches.Patch(color=c,label=b.replace("Block","")) for b,c in BLOCK_COLORS.items()]
axes[-1].legend(handles=patches,loc="lower center",ncol=6,fontsize=7,bbox_to_anchor=(.5,-.45))
fig.suptitle("Architecture Chromosomes — 5 Random Genotypes\n"
             "(Grey=global · Colored=stage blocks · Orange=head)",
             fontsize=12,fontweight="bold")
plt.savefig("fig3_chromosomes.png",dpi=150,bbox_inches="tight"); plt.show()

In [ ]:
# ── Block gallery ─────────────────────────────────────────────────────────────
BLOCK_DESC = {
    "ConvBlock":          ("Conv+BN+Act + skip","General feature extraction"),
    "SepConvBlock":       ("DW + PW conv","Parameter efficient"),
    "ResidualBlock":      ("2× conv + identity","Stable deep networks"),
    "MBConvBlock":        ("Expand→DW→project","Mobile inverted residuals"),
    "BottleneckBlock":    ("1×1→k×k→1×1","High-channel representations"),
    "AnisotropicBlock":   ("1×k + k×1 factored","Elongated / sequence inputs"),
    "DilatedConvBlock":   ("Dilated conv d=2..4","Large RF without stride"),
    "GridLogicBlock":     ("Local+pointwise mix","Rule-based grids"),
    "ChannelMixingBlock": ("1×1 bottleneck","Channel-heavy inputs"),
    "GlobalContextBlock": ("Local + pool bias","Global context everywhere"),
    "LightAttentionBlock":("MHA spatial tokens","Small spatial (H·W≤256)"),
}
fig,axes = plt.subplots(3,4,figsize=(16,8),gridspec_kw={"hspace":.55,"wspace":.2})
for i,(name,(arch,use)) in enumerate(BLOCK_DESC.items()):
    ax = axes.flatten()[i]; ax.axis("off")
    col = BLOCK_COLORS[name]
    ax.add_patch(mpatches.FancyBboxPatch((0,0),1,1,boxstyle="round,pad=.04",
        facecolor=col+"18",edgecolor=col,linewidth=2,transform=ax.transAxes,clip_on=False))
    ax.add_patch(mpatches.FancyBboxPatch((0,.72),1,.28,boxstyle="round,pad=.01",
        facecolor=col,edgecolor="none",transform=ax.transAxes,clip_on=True))
    ax.text(.5,.86,name.replace("Block",""),ha="center",va="center",
            transform=ax.transAxes,fontsize=10,fontweight="bold",color="white")
    ax.text(.5,.57,arch,ha="center",va="center",transform=ax.transAxes,
            fontsize=8,color="#444",style="italic")
    ax.text(.5,.3,use,ha="center",va="center",transform=ax.transAxes,
            fontsize=9,color=col,fontweight="bold")
axes.flatten()[-1].axis("off")
fig.suptitle("Primitive Block Library  (each composable with SEBlock + DropPath)",
             fontsize=14,fontweight="bold")
plt.savefig("fig4_blocks.png",dpi=150,bbox_inches="tight"); plt.show()

---
## ◆ Section 4 — AZ-NAS: Zero-Cost Proxy

Training each candidate for even a few epochs is expensive. **AZ-NAS** (CVPR 2024) evaluates
architecture quality from a **single forward+backward pass** — no weight training needed.

$$\text{AZ-NAS} = \underbrace{E}_{\text{Expressivity}} + \underbrace{P}_{\text{Progressivity}} + \underbrace{T}_{\text{Trainability}} - \lambda\,\underbrace{C}_{\log(\text{params})}$$

- **E** — entropy of eigenvalue spectrum of per-layer spatial covariance → feature diversity
- **P** — minimum inter-layer expressivity delta → network must deepen, not flatten
- **T** — singular value analysis of inter-layer Jacobian → gradient flow quality
- **C** — log(params) soft penalty for oversized models

In [ ]:
from search_space import az_nas_score_full, build_model
from search_space.proxies import _init_model, _expressivity_score

random.seed(7)
g0 = sample_random_genotype()
g0 = repair(g0,C,H,W,NUM_CLASSES,OUR_FAM)
model0 = build_model(g0,C,H,W,NUM_CLASSES,aniso_axis=OUR_FAM.aniso_axis)

x_batch = torch.tensor(train_x[:32],dtype=torch.float32)
_init_model(model0)
model0.train()
with torch.no_grad():
    layer_feats = model0.extract_layer_features(x_batch)
layer_names = ["Stem"]+[f"Stage {i+1}" for i in range(len(layer_feats)-1)]
expr_scores = [_expressivity_score(f) for f in layer_feats]

banner("§4  AZ-NAS Zero-Cost Proxy",
       "Expressivity · Progressivity · Trainability · Complexity",PALETTE["orange"])

fig = plt.figure(figsize=(16,9))
gs  = gridspec.GridSpec(2,3,figure=fig,hspace=.48,wspace=.38)

# Expressivity progression ──────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0,0])
ax1.plot(range(len(expr_scores)),expr_scores,"o-",color=PALETTE["blue"],lw=2.5,ms=8,zorder=3)
ax1.fill_between(range(len(expr_scores)),expr_scores,alpha=.15,color=PALETTE["blue"])
for i,(nm,sc) in enumerate(zip(layer_names,expr_scores)):
    ax1.annotate(f"{sc:.1f}",(i,sc),textcoords="offset points",xytext=(0,10),
                 ha="center",fontsize=8,color=PALETTE["blue"])
ax1.set_xticks(range(len(layer_names)))
ax1.set_xticklabels(layer_names,rotation=20,ha="right",fontsize=9)
ax1.set_ylabel("Expressivity $E_i$")
ax1.set_title("Layer-wise Expressivity\n(want: monotone increasing)")
deltas = np.diff(expr_scores)
pc = PALETTE["red"] if deltas.min()<0 else PALETTE["green"]
ax1.text(.98,.05,f"P = {deltas.min():.2f}",transform=ax1.transAxes,ha="right",
         fontsize=9,color=pc,fontweight="bold",
         bbox=dict(boxstyle="round",facecolor=pc+"22",edgecolor=pc))

# Eigenvalue spectrum ─────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0,1])
feat = layer_feats[-1].detach()
b,cf,hf,wf = feat.shape
flat = feat.permute(0,2,3,1).reshape(-1,cf)
flat = flat - flat.mean(0)
sigma = (flat.T @ flat)/flat.shape[0]
eigvals = torch.linalg.eigvalsh(sigma).clamp(min=0).numpy()[::-1]
prob = eigvals/(eigvals.sum()+1e-8)
cum_var = np.cumsum(prob)
ax2t = ax2.twinx()
ax2.bar(range(min(32,len(eigvals))),eigvals[:32],color=PALETTE["teal"],alpha=.75)
ax2t.plot(range(min(32,len(cum_var))),cum_var[:32]*100,color=PALETTE["orange"],lw=2)
ax2.set_xlabel("Eigenvalue rank"); ax2.set_ylabel("Magnitude",color=PALETTE["teal"])
ax2t.set_ylabel("Cumulative var (%)",color=PALETTE["orange"])
ax2.set_title("Eigenvalue Spectrum (last layer)")
H_val = (-prob*np.log(prob+1e-8)).sum()
ax2.text(.97,.95,f"H = {H_val:.2f}",transform=ax2.transAxes,ha="right",va="top",
         fontsize=9,color=PALETTE["teal"],fontweight="bold")

# Feature map thumbnails ─────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0,2]); ax3.axis("off")
CMAPS = ["inferno","viridis","plasma","cividis"]
inner3 = gridspec.GridSpecFromSubplotSpec(
    len(layer_feats),4,subplot_spec=gs[0,2],hspace=.06,wspace=.06)
for li,feat in enumerate(layer_feats):
    fmap = feat[0].detach().numpy()
    for ci in range(min(4,fmap.shape[0])):
        axf = fig.add_subplot(inner3[li,ci])
        m = fmap[ci]; m = (m-m.min())/(m.max()-m.min()+1e-8)
        axf.imshow(m,cmap=CMAPS[ci],interpolation="nearest")
        axf.set_xticks([]); axf.set_yticks([])
        if ci==0: axf.set_ylabel(layer_names[li],fontsize=7,rotation=0,labelpad=35,va="center")
ax3.set_title("Feature Maps (4ch per layer)",pad=80)

# Component bars for 3 archs ─────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1,:2])
random.seed(0)
eval_scores = []
for i in range(3):
    gx = sample_random_genotype(); gx = repair(gx,C,H,W,NUM_CLASSES,OUR_FAM)
    mx = build_model(gx,C,H,W,NUM_CLASSES,aniso_axis=OUR_FAM.aniso_axis)
    eval_scores.append(az_nas_score_full(mx,x_batch,"cpu",reinit=True))
x_pos=np.arange(4); w=.22
KEYS=["expressivity","progressivity","trainability","complexity"]
for i,(sc,col) in enumerate(zip(eval_scores,[PALETTE["blue"],PALETTE["orange"],PALETTE["teal"]])):
    ax4.bar(x_pos+i*w,[sc[k] for k in KEYS],w,label=f"Arch {i+1}  az={sc["az_nas"]:.1f}",
            color=col,alpha=.85,edgecolor="white")
ax4.set_xticks(x_pos+w); ax4.set_xticklabels(["Expr. E","Prog. P","Train. T","Compl. C"],fontsize=10)
ax4.set_ylabel("Score"); ax4.set_title("AZ-NAS Components — 3 Random Architectures")
ax4.legend(fontsize=9); ax4.axhline(0,color="#999",lw=.8)

# Formula callout ─────────────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1,2]); ax5.axis("off")
ax5.add_patch(mpatches.FancyBboxPatch((.05,.08),.9,.84,boxstyle="round,pad=.04",
    facecolor=PALETTE["purple"]+"15",edgecolor=PALETTE["purple"],lw=2))
for y_t,txt,col,fs in [(.88,"AZ-NAS Formula",PALETTE["purple"],13),
    (.73,"score = E + P + T − λ·C","#333",12),(.60,"λ = 0.1","#555",11),
    (.47,"E = Σ H(eigenvalues σ)",PALETTE["blue"],10),
    (.37,"P = min Δ(Eᵢ)",PALETTE["orange"],10),
    (.27,"T = mean(−σmax−1/σmax+2)",PALETTE["teal"],9),
    (.17,"C = log(params)",PALETTE["red"],10)]:
    ax5.text(.5,y_t,txt,ha="center",va="center",transform=ax5.transAxes,
             fontsize=fs,color=col,fontweight="bold" if y_t>.8 else "normal")

fig.suptitle("AZ-NAS Zero-Cost Proxy — Deep Dive on Real Data",
             fontsize=15,fontweight="bold",y=1.01)
plt.savefig("fig5_aznas.png",dpi=150,bbox_inches="tight"); plt.show()

---
## ◆ Section 5 — Population Landscape
Sample 30 random architectures, score each with AZ-NAS, analyze the score landscape.

In [ ]:
banner("§5  Population Landscape","30 random architectures scored with AZ-NAS",PALETTE["green"])

N_POP = 30
random.seed(123)
pop_results = []
print(f"Evaluating {N_POP} random architectures on {DS_NAME}...")
for i in range(N_POP):
    g = sample_random_genotype()
    g = repair(g,C,H,W,NUM_CLASSES,OUR_FAM)
    try:
        m = build_model(g,C,H,W,NUM_CLASSES,aniso_axis=OUR_FAM.aniso_axis)
        s = az_nas_score_full(m,x_batch,"cpu",reinit=True)
        p = sum(x.numel() for x in m.parameters())
        pop_results.append({
            "idx":i,"params":p,"n_stages":g.n_stages,"block":g.stages[0].block_type,
            "head":g.head_type,**s,
        })
        mk = " ★" if s["az_nas"]==max(r["az_nas"] for r in pop_results) else ""
        print(f"  [{i+1:>2}/{N_POP}]  az={s["az_nas"]:7.2f}  E={s["expressivity"]:6.2f}  P={s["progressivity"]:6.2f}  T={s["trainability"]:6.2f}  {p/1e6:.1f}M{mk}")
    except Exception as e: print(f"  [{i+1:>2}/{N_POP}]  SKIP: {e}")

pop_results = [r for r in pop_results if np.isfinite(r["az_nas"])]
pop_results.sort(key=lambda r:r["az_nas"],reverse=True)
print(f"\n✓  {len(pop_results)} valid  |  best={pop_results[0]["az_nas"]:.3f}  worst={pop_results[-1]["az_nas"]:.3f}")

In [ ]:
fig = plt.figure(figsize=(16,10))
gs  = gridspec.GridSpec(2,3,figure=fig,hspace=.45,wspace=.38)

sc_all = [r["az_nas"] for r in pop_results]
ex_all = [r["expressivity"] for r in pop_results]
pr_all = [r["progressivity"] for r in pop_results]
tr_all = [r["trainability"] for r in pop_results]
pm_all = [r["params"]/1e6 for r in pop_results]

# Ranked bar ──────────────────────────────────────────────────────────────────
ax1=fig.add_subplot(gs[0,0])
cols_bar=[PALETTE["gold"] if i==0 else PALETTE["blue"]+"aa" for i in range(len(sc_all))]
ax1.bar(range(len(sc_all)),sc_all,color=cols_bar,edgecolor="white",lw=.5)
ax1.set_xlabel("Architecture rank"); ax1.set_ylabel("AZ-NAS Score")
ax1.set_title("Population Ranked by Score")
ax1.text(0,sc_all[0]+.3,"★ Best",ha="left",fontsize=8,color=PALETTE["gold"],fontweight="bold")

# Params vs score ─────────────────────────────────────────────────────────────
ax2=fig.add_subplot(gs[0,1])
sc2=ax2.scatter(pm_all,sc_all,c=sc_all,cmap="RdYlGn",s=80,edgecolors="#555",lw=.5,zorder=3)
plt.colorbar(sc2,ax=ax2,shrink=.8,label="AZ-NAS Score")
ax2.set_xlabel("Parameters (M)"); ax2.set_ylabel("AZ-NAS Score")
ax2.set_title("Params vs Score (Pareto view)")
br = pop_results[0]
ax2.annotate("★ Best",(br["params"]/1e6,br["az_nas"]),xytext=(10,-20),
             textcoords="offset points",arrowprops=dict(arrowstyle="->",color=PALETTE["orange"]),
             fontsize=9,color=PALETTE["orange"],fontweight="bold")

# Score distributions ─────────────────────────────────────────────────────────
ax3=fig.add_subplot(gs[0,2])
vp=ax3.violinplot([ex_all,pr_all,tr_all,sc_all],positions=[1,2,3,4],showmedians=True)
for pc,col in zip(vp["bodies"],[PALETTE["blue"],PALETTE["orange"],PALETTE["teal"],PALETTE["purple"]]):
    pc.set_facecolor(col); pc.set_alpha(.7)
ax3.set_xticks([1,2,3,4])
ax3.set_xticklabels(["Expr.","Prog.","Train.","AZ-NAS"],fontsize=10)
ax3.set_ylabel("Score"); ax3.set_title("Score Distributions (violin)")
ax3.axhline(0,ls="--",color="#aaa",lw=1)

# By n_stages ──────────────────────────────────────────────────────────────────
ax4=fig.add_subplot(gs[1,0])
for ns in sorted(set(r["n_stages"] for r in pop_results)):
    sg=[r["az_nas"] for r in pop_results if r["n_stages"]==ns]
    ax4.scatter([ns]*len(sg),sg,alpha=.6,s=50,label=f"{ns} stages")
    ax4.plot([ns-.2,ns+.2],[np.mean(sg)]*2,"k-",lw=2)
ax4.set_xlabel("Number of stages"); ax4.set_ylabel("AZ-NAS"); ax4.set_title("Score by Depth")
ax4.legend(fontsize=8)

# By block type ───────────────────────────────────────────────────────────────
ax5=fig.add_subplot(gs[1,1])
blks=sorted(set(r["block"] for r in pop_results))
bsc={b:[r["az_nas"] for r in pop_results if r["block"]==b] for b in blks}
means=[np.mean(v) for v in bsc.values()]
order=np.argsort(means)[::-1]
for rank,idx in enumerate(order):
    b=blks[idx]; v=bsc[b]
    ax5.barh(rank,np.mean(v),xerr=np.std(v) if len(v)>1 else 0,
             color=BLOCK_COLORS.get(b,"#888"),alpha=.85,capsize=3)
    ax5.text(.5,rank,b.replace("Block","")[:12],va="center",fontsize=7.5,
             color="white",fontweight="bold")
ax5.set_yticks([]); ax5.set_xlabel("Mean AZ-NAS ± std")
ax5.set_title("Score by Stage-1 Block Type")

# Leaderboard table ───────────────────────────────────────────────────────────
ax6=fig.add_subplot(gs[1,2]); ax6.axis("off")
top5=pop_results[:5]
tbl=ax6.table(
    cellText=[[f"#{i+1}",f"{r["az_nas"]:.2f}",f"{r["expressivity"]:.2f}",
               f"{r["progressivity"]:.2f}",f"{r["trainability"]:.2f}",
               f"{r["params"]/1e6:.1f}M"] for i,r in enumerate(top5)],
    colLabels=["Rank","AZ-NAS","Expr.","Prog.","Train.","Params"],
    cellLoc="center",loc="center",bbox=[0,.1,1,.85])
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
for (row,col),cell in tbl.get_celld().items():
    if row==0: cell.set_facecolor(PALETTE["purple"]); cell.set_text_props(color="white",fontweight="bold")
    elif row==1: cell.set_facecolor(PALETTE["gold"]+"55")
    else: cell.set_facecolor("#F5F5F5" if row%2 else "white")
    cell.set_edgecolor("#DDD")
ax6.set_title("Top-5 Leaderboard",fontsize=11,fontweight="bold")

fig.suptitle(f"Population Landscape — {len(pop_results)} Architectures on {DS_NAME}",
             fontsize=14,fontweight="bold",y=1.01)
plt.savefig("fig6_population.png",dpi=150,bbox_inches="tight"); plt.show()

---
## ◆ Section 6 — Aging Evolution

**Aging Evolution** (Real et al. 2019) improves the population by:
1. Tournament selection → pick best parent
2. Mutate (scale anneals: large→medium→small)
3. Repair → enforce hard constraints
4. Evaluate with AZ-NAS (~0.1s per arch)
5. Add child, evict **oldest** (not worst) — keeps diversity high

In [ ]:
from search_space import aging_evolution, best_individual, az_nas_score

banner("§6  Aging Evolution",
       "Tournament · Age-based eviction · Adaptive mutation scale",PALETTE["red"])

def proxy_fn(model, batch_x, dev):
    return az_nas_score(model, batch_x, dev, reinit=True)

N_POP_EVO = 25
N_ROUNDS  = 60
random.seed(42)
print(f"Running Aging Evolution: pop={N_POP_EVO}, rounds={N_ROUNDS}...")
t0 = time.time()
population = aging_evolution(
    family=OUR_FAM, C=C, H=H, W=W, num_classes=NUM_CLASSES,
    proxy_fn=proxy_fn, batch_x=x_batch, device="cpu",
    n_population=N_POP_EVO, n_rounds=N_ROUNDS,
    tournament_size=7, time_budget_s=180, verbose=True,
)
elapsed = time.time()-t0
best = best_individual(population)
print(f"\n✓ Done in {elapsed:.1f}s  |  best={best.fitness:.4f}")
print(f"  Stages: {best.genotype.n_stages}  Head: {best.genotype.head_type}  Norm: {best.genotype.norm_type}")

In [ ]:
fig,axes = plt.subplots(1,3,figsize=(16,5))
pop_fit = sorted([ind.fitness for ind in population if np.isfinite(ind.fitness)],reverse=True)
rand_sc = sorted(sc_all,reverse=True)

# Fitness bars ────────────────────────────────────────────────────────────────
ax1=axes[0]
cols_e=[PALETTE["gold"] if i==0 else PALETTE["red"]+"bb" for i in range(len(pop_fit))]
ax1.bar(range(len(pop_fit)),pop_fit,color=cols_e,edgecolor="white",lw=.5)
ax1.axhline(np.mean(rand_sc),ls="--",color=PALETTE["grey"],lw=1.5,
            label=f"Random mean: {np.mean(rand_sc):.1f}")
ax1.axhline(max(rand_sc),ls=":",color=PALETTE["grey"],lw=1.5,
            label=f"Random best: {max(rand_sc):.1f}")
ax1.set_xlabel("Rank"); ax1.set_ylabel("AZ-NAS Score")
ax1.set_title("Evolved Population"); ax1.legend(fontsize=8)

# Evolution vs random ─────────────────────────────────────────────────────────
ax2=axes[1]
ax2.hist(rand_sc,bins=10,alpha=.65,color=PALETTE["grey"],label=f"Random (n={len(rand_sc)})",edgecolor="white")
ax2.hist(pop_fit,bins=10,alpha=.65,color=PALETTE["red"],label=f"Evolved (n={len(pop_fit)})",edgecolor="white")
ax2.axvline(max(rand_sc),ls="--",color=PALETTE["grey"],lw=1.5)
ax2.axvline(max(pop_fit),ls="--",color=PALETTE["red"],lw=1.5)
ax2.set_xlabel("AZ-NAS Score"); ax2.set_ylabel("Count")
ax2.set_title("Evolution vs Random Sampling"); ax2.legend(fontsize=9)
delta = max(pop_fit)-max(rand_sc)
ax2.text(.97,.93,f"Δ best = +{delta:.2f}",transform=ax2.transAxes,ha="right",va="top",
         fontsize=10,color=PALETTE["red"],fontweight="bold",
         bbox=dict(boxstyle="round",facecolor=PALETTE["red"]+"22",edgecolor=PALETTE["red"]))

# Radar chart ─────────────────────────────────────────────────────────────────
ax3=fig.add_subplot(1,3,3,polar=True)
RAD_KEYS=["expressivity","progressivity","trainability","az_nas"]
N_r=4; angles=np.linspace(0,2*np.pi,N_r,endpoint=False).tolist(); angles+=angles[:1]
def norm(vals):
    result=[]
    for k,v in zip(RAD_KEYS,vals):
        av=[r[k] for r in pop_results if np.isfinite(r[k])]
        mn,mx=min(av),max(av)
        result.append((v-mn)/(mx-mn+1e-8))
    return result+[result[0]]
b_r=pop_results[0]; m_r=pop_results[len(pop_results)//2]
bv=norm([b_r[k] for k in RAD_KEYS]); mv=norm([m_r[k] for k in RAD_KEYS])
ax3.plot(angles,bv,"o-",color=PALETTE["red"],lw=2,label="Best evolved")
ax3.fill(angles,bv,alpha=.25,color=PALETTE["red"])
ax3.plot(angles,mv,"s--",color=PALETTE["grey"],lw=1.5,label="Median")
ax3.fill(angles,mv,alpha=.1,color=PALETTE["grey"])
ax3.set_xticks(angles[:-1])
ax3.set_xticklabels(["Expr.","Prog.","Train.","AZ-NAS"],fontsize=10)
ax3.set_ylim(0,1); ax3.set_yticks([])
ax3.legend(loc="upper right",bbox_to_anchor=(1.35,1.15),fontsize=9)
ax3.set_title("Best vs Median\n(normalized)",pad=20,fontsize=11)

fig.suptitle("Aging Evolution Results",fontsize=14,fontweight="bold",y=1.01)
plt.savefig("fig7_evolution.png",dpi=150,bbox_inches="tight"); plt.show()

---
## ◆ Section 7 — Proxy ↔ Accuracy Correlation

**Key validation**: does AZ-NAS rank architectures the same way 3-epoch training does?
A high Spearman ρ confirms the proxy is a reliable fitness signal.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import spearmanr

banner("§7  Proxy–Accuracy Correlation",
       "Does AZ-NAS predict which architectures will learn?",PALETTE["gold"])

x_t = torch.tensor(train_x,dtype=torch.float32)
y_t = torch.tensor(train_y,dtype=torch.long)
split = int(.8*len(x_t))
tr_dl = DataLoader(TensorDataset(x_t[:split],y_t[:split]),batch_size=32,shuffle=True)
va_dl = DataLoader(TensorDataset(x_t[split:],y_t[split:]),batch_size=32)

def short_train(model,n_epochs=3):
    model.to("cpu").train()
    opt=torch.optim.Adam(model.parameters(),lr=3e-3)
    crit=torch.nn.CrossEntropyLoss()
    for _ in range(n_epochs):
        for xb,yb in tr_dl:
            opt.zero_grad(); crit(model(xb),yb).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        ps=[]; ls=[]
        for xb,yb in va_dl: ps+=model(xb).argmax(1).tolist(); ls+=yb.tolist()
    return accuracy_score(ls,ps)

all_r = sorted(pop_results,key=lambda r:r["az_nas"])
idxs  = np.linspace(0,len(all_r)-1,min(10,len(all_r)),dtype=int)
sampled = [all_r[i] for i in idxs]

corr_results = []
print("Training 10 architectures for 3 epochs each...")
for j,r in enumerate(sampled):
    random.seed(r["idx"]); np.random.seed(r["idx"])
    gx=sample_random_genotype(); gx=repair(gx,C,H,W,NUM_CLASSES,OUR_FAM)
    try:
        mx=build_model(gx,C,H,W,NUM_CLASSES,aniso_axis=OUR_FAM.aniso_axis)
        acc=short_train(copy.deepcopy(mx),n_epochs=3)
        corr_results.append({"proxy":r["az_nas"],"acc":acc,"params":r["params"]})
        print(f"  [{j+1:>2}/10]  proxy={r["az_nas"]:8.3f}  val_acc={acc:.3f}")
    except Exception as e: print(f"  [{j+1:>2}/10]  SKIP: {e}")
print(f"\n✓  {len(corr_results)} data points")

In [ ]:
if len(corr_results)>=4:
    proxies=[r["proxy"] for r in corr_results]
    accs   =[r["acc"]*100 for r in corr_results]
    rho,pval=spearmanr(proxies,accs)

    fig,axes=plt.subplots(1,2,figsize=(14,5.5))

    ax1=axes[0]
    sc=ax1.scatter(proxies,accs,c=[r["params"]/1e6 for r in corr_results],
                   cmap="YlOrRd",s=120,edgecolors="#555",lw=.8,zorder=3)
    plt.colorbar(sc,ax=ax1,label="Params (M)",shrink=.8)
    for i,(px,ac) in enumerate(zip(proxies,accs)):
        ax1.annotate(f"{corr_results[i]["params"]/1e6:.1f}M",(px,ac),
                     textcoords="offset points",xytext=(5,4),fontsize=7.5,color="#555")
    z=np.polyfit(proxies,accs,1); xs=np.linspace(min(proxies),max(proxies),80)
    ax1.plot(xs,np.poly1d(z)(xs),"--",color=PALETTE["red"],lw=2,alpha=.7,label="linear fit")
    ax1.set_xlabel("AZ-NAS Proxy Score",fontsize=12)
    ax1.set_ylabel("Val Accuracy after 3 epochs (%)",fontsize=12)
    ax1.set_title("Zero-Cost Proxy vs Training Accuracy"); ax1.legend(fontsize=9)
    badge=PALETTE["green"] if rho>.5 else (PALETTE["gold"] if rho>.2 else PALETTE["red"])
    ax1.text(.04,.96,f"Spearman ρ = {rho:.3f}\np-value = {pval:.3f}",
             transform=ax1.transAxes,ha="left",va="top",fontsize=11,fontweight="bold",
             color=badge,bbox=dict(boxstyle="round,pad=.5",facecolor=badge+"22",edgecolor=badge,lw=2))

    ax2=axes[1]
    pr_rnk=np.argsort(np.argsort(proxies)); ac_rnk=np.argsort(np.argsort([r["acc"] for r in corr_results]))
    n=len(proxies); y_pos=np.arange(n)
    ax2.barh(y_pos-.2,pr_rnk,.38,color=PALETTE["blue"],alpha=.8,label="Proxy rank")
    ax2.barh(y_pos+.2,ac_rnk,.38,color=PALETTE["orange"],alpha=.8,label="Accuracy rank")
    for i in range(n): ax2.plot([pr_rnk[i],ac_rnk[i]],[i-.2,i+.2],"k-",lw=1.2,alpha=.5)
    ax2.set_yticks(y_pos); ax2.set_yticklabels([f"Arch {i+1}" for i in range(n)],fontsize=9)
    ax2.set_xlabel("Rank"); ax2.set_title("Proxy Rank vs Accuracy Rank")
    ax2.legend(fontsize=9)

    msg=("✓ Strong — proxy reliably selects good architectures." if rho>.6
         else "~ Moderate — useful signal, not perfect." if rho>.3
         else "⚠ Weak — proxy may need tuning for this dataset.")
    fig.suptitle(f"Proxy–Accuracy Correlation  |  {msg}",fontsize=12,fontweight="bold",y=1.02)
    plt.savefig("fig8_correlation.png",dpi=150,bbox_inches="tight"); plt.show()

---
## ◆ Section 8 — Full Pipeline (Mini End-to-End)

> **⚠️ WARNING — READ BEFORE RUNNING ⚠️**
>
> This section runs the **complete** DataProcessor → NAS → Trainer pipeline.
> - On **CPU without GPU**: may take **30–90 minutes** per dataset.
> - On **GPU (Colab T4)**: ~5–15 minutes per dataset.
> - With **real datasets + full time budget**: potentially **hours**.
>
> The cell below is configured with a **3-minute cap** (`time_limit: 0.05h`) and synthetic data
> so it finishes in ~2 minutes. To run on real data, change `time_limit` and `DS_NAME`.
>
> **Do NOT run this on a real dataset without a GPU unless you have patience and coffee ☕**

In [ ]:
# ┌──────────────────────────────────────────────────────────────────────────┐
# │  ⚠  FULL PIPELINE — MINI VERSION (3 min cap, synthetic data)            │
# │  To use real data: change time_limit → 0.5 and use a real dataset       │
# └──────────────────────────────────────────────────────────────────────────┘

banner("§8  Full Pipeline (Mini)","⚠  Capped at 3 minutes — see warnings above",PALETTE["dark"])

from torch.utils.data import DataLoader, TensorDataset

# ── Tiny synthetic dataset (fast) ────────────────────────────────────────────
PIPE_C, PIPE_H, PIPE_W, PIPE_NC = 3, 28, 28, 10
PIPE_N = 256

px_train = torch.tensor(train_x[:int(PIPE_N*.7)], dtype=torch.float32)
py_train = torch.tensor(train_y[:int(PIPE_N*.7)], dtype=torch.long)
px_val   = torch.tensor(train_x[int(PIPE_N*.7):PIPE_N], dtype=torch.float32)
py_val   = torch.tensor(train_y[int(PIPE_N*.7):PIPE_N], dtype=torch.long)
px_test  = torch.tensor(train_x[PIPE_N:PIPE_N+50], dtype=torch.float32)
py_test  = torch.tensor(train_y[PIPE_N:PIPE_N+50], dtype=torch.long)

pipe_train = DataLoader(TensorDataset(px_train,py_train),batch_size=32,shuffle=True)
pipe_val   = DataLoader(TensorDataset(px_val,  py_val),  batch_size=32)
pipe_test  = DataLoader(TensorDataset(px_test, py_test), batch_size=32)

pipe_meta = {
    "input_shape": [PIPE_N,PIPE_C,PIPE_H,PIPE_W],
    "num_classes": NUM_CLASSES,
    "benchmark":   BENCHMARK,
    "codename":    DS_NAME,
    "time_limit":  0.05,     # ← 3 minutes total
}

# ── Mock clock ────────────────────────────────────────────────────────────────
class MockClock:
    def __init__(self,seconds=180):
        self._start=time.perf_counter(); self._budget=seconds
    def check(self): return max(0.,self._budget-(time.perf_counter()-self._start))

clock = MockClock(seconds=180)

# ── Step 1: NAS ───────────────────────────────────────────────────────────────
from nas import NAS
print("="*60)
print("STEP 1: NAS — searching for best architecture...")
print("="*60)
t0=time.time()
model = NAS(pipe_train,pipe_val,pipe_meta,clock).search()
print(f"NAS: {time.time()-t0:.1f}s  |  {type(model).__name__}  |  {sum(p.numel() for p in model.parameters()):,} params")

# ── Step 2: Train ─────────────────────────────────────────────────────────────
from trainer import Trainer
print("\n"+"="*60)
print("STEP 2: Trainer — training until time runs out...")
print("="*60)
t0=time.time()
trainer=Trainer(model,DEVICE,pipe_train,pipe_val,pipe_meta,clock)
trainer.train()
print(f"Training: {time.time()-t0:.1f}s")

# ── Step 3: Predict & evaluate ───────────────────────────────────────────────
print("\n"+"="*60)
print("STEP 3: Predict & evaluate")
print("="*60)
preds = trainer.predict(pipe_test)
acc   = accuracy_score(py_test.numpy(), preds)
baseline = 1.0/NUM_CLASSES
adj  = (acc*100-BENCHMARK)*10/(100-BENCHMARK)
print(f"Test accuracy : {acc*100:.2f}%")
print(f"Random baseline: {baseline*100:.2f}%")
print(f"Benchmark      : {BENCHMARK:.2f}%")
print(f"Adj Score      : {adj:+.3f}  (0=benchmark, 10=perfect)")
print("\n✓  PIPELINE COMPLETE")

---
## ◆ Section 9 — Summary Dashboard

In [ ]:
banner("§9  Summary","Results across all experiment sections",PALETTE["dark"])

fig,axes=plt.subplots(1,3,figsize=(16,6))

# Pipeline card ────────────────────────────────────────────────────────────────
ax1=axes[0]; ax1.axis("off")
g_best = best.genotype
m_best = build_model(g_best,C,H,W,NUM_CLASSES,aniso_axis=OUR_FAM.aniso_axis)
n_best = sum(p.numel() for p in m_best.parameters())
s_best = az_nas_score_full(m_best,x_batch,"cpu",reinit=True)
steps=[
    (PALETTE["blue"],  "DATA",   f"{DS_NAME}\n{C}×{H}×{W} · {NUM_CLASSES} cls"),
    (PALETTE["teal"],  "FAMILY", f"{OUR_FAM.name}\nmax_pools={OUR_FAM.max_pool_steps}"),
    (PALETTE["purple"],"SPACE",  "11 blocks · 7 heads"),
    (PALETTE["orange"],"AZ-NAS", "E+P+T−λC\n~0.1s/arch"),
    (PALETTE["red"],   "EVOLVE", f"pop={N_POP_EVO} · {N_ROUNDS} rounds"),
    (PALETTE["green"], "RESULT", f"score={best.fitness:.2f}\n{n_best/1e6:.1f}M params"),
]
for k,(col,title,body) in enumerate(steps):
    y=.97-k*.162
    ax1.add_patch(mpatches.FancyBboxPatch((.02,y-.13),.96,.125,
        boxstyle="round,pad=.02",facecolor=col+"22",edgecolor=col,lw=1.5))
    ax1.add_patch(mpatches.FancyBboxPatch((.02,y-.13),.22,.125,
        boxstyle="round,pad=.02",facecolor=col,edgecolor="none"))
    ax1.text(.12,y-.065,title,ha="center",va="center",fontsize=8,color="white",fontweight="bold")
    ax1.text(.62,y-.065,body,ha="center",va="center",fontsize=9,color="#333")
ax1.set_xlim(0,1); ax1.set_ylim(0,1)
ax1.set_title("Pipeline Overview",fontsize=12,fontweight="bold")

# Metrics table ────────────────────────────────────────────────────────────────
ax2=axes[1]; ax2.axis("off")
met=[
    ("Dataset",DS_NAME),("Shape",f"{C}×{H}×{W}"),
    ("Family",OUR_FAM.name.replace("_"," ")),("Benchmark",f"{BENCHMARK:.1f}%"),
    ("Architectures",f"{len(pop_results)} random + {N_ROUNDS} evolved"),
    ("Best proxy",f"{best.fitness:.3f}"),
    ("Random best",f"{max(sc_all):.3f}"),
    ("Improvement",f"+{best.fitness-max(sc_all):.3f}"),
    ("Best params",f"{n_best/1e6:.2f}M"),
    ("Best head",g_best.head_type),
]
if len(corr_results)>=4: met.append(("Proxy ρ",f"{rho:.3f} (p={pval:.3f})"))
tbl=ax2.table(cellText=[[k,v] for k,v in met],colLabels=["Metric","Value"],
              cellLoc="left",loc="center",bbox=[0,0,1,1])
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
for (row,col),cell in tbl.get_celld().items():
    if row==0: cell.set_facecolor(PALETTE["dark"]); cell.set_text_props(color="white",fontweight="bold")
    elif col==0: cell.set_facecolor("#ECEFF1")
    else: cell.set_facecolor("#E8F5E9" if row==8 else "white")
    cell.set_edgecolor("#CCC")
ax2.set_title("Key Results",fontsize=12,fontweight="bold")

# Radar ────────────────────────────────────────────────────────────────────────
ax3=fig.add_subplot(1,3,3,polar=True)
bv_r=norm([s_best[k] for k in RAD_KEYS])
mr=pop_results[len(pop_results)//2]
mv_r=norm([mr[k] for k in RAD_KEYS])
ax3.plot(angles,bv_r,"o-",color=PALETTE["red"],lw=2,label="Best evolved")
ax3.fill(angles,bv_r,alpha=.25,color=PALETTE["red"])
ax3.plot(angles,mv_r,"s--",color=PALETTE["grey"],lw=1.5,label="Median")
ax3.fill(angles,mv_r,alpha=.1,color=PALETTE["grey"])
ax3.set_xticks(angles[:-1]); ax3.set_xticklabels(["Expr.","Prog.","Train.","AZ-NAS"],fontsize=10)
ax3.set_ylim(0,1); ax3.set_yticks([])
ax3.legend(loc="upper right",bbox_to_anchor=(1.35,1.15),fontsize=9)
ax3.set_title("Best vs Median (normalized)",pad=20,fontsize=11)

fig.suptitle("Experiment Summary — Zero-Cost NAS on Heterogeneous NCHW Tensors",
             fontsize=14,fontweight="bold",y=1.02)
plt.savefig("fig9_summary.png",dpi=150,bbox_inches="tight"); plt.show()

print("\n"+"="*60)
print("EXPERIMENT COMPLETE")
print(f"  Dataset:     {DS_NAME}  ({DATA_SOURCE} data)")
print(f"  Family:      {OUR_FAM.name}")
print(f"  Best score:  {best.fitness:.4f}  (random best: {max(sc_all):.4f})")
print(f"  Improvement: +{best.fitness-max(sc_all):.4f}")
print(f"  Saved:  fig1…fig9 as PNG")
print("="*60)